In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer

TARGET_COL = "y"
TEST_SIZE = 0.2
RANDOM_STATE = 42

df = pd.read_csv(r"data/data.csv", sep=";")
print(f"Dataset shape: {df.shape}")
df.head(3)

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()


Dataset shape: (45211, 17)


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numerical columns:   {len(numerical_cols)}")

Train shape: (36168, 16)
Test shape:  (9043, 16)
Categorical columns: 9
Numerical columns:   7


C:\Users\gvinicir\AppData\Local\Temp\ipykernel_18456\1963129725.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


In [21]:
# 1) Turn 'unknown' into NaN for categorical columns
for col in categorical_cols:
    X_train[col] = X_train[col].replace("unknown", np.nan)
    X_test[col] = X_test[col].replace("unknown", np.nan)


In [22]:
# 2) Mark top 0.1% absolute z-score values as outliers (set to NaN)
zscore_info = {}

for col in numerical_cols:
    mean_col = X_train[col].mean()
    std_col = X_train[col].std(ddof=0)

    if pd.isna(std_col) or std_col == 0:
        zscore_info[col] = {
            "mean": mean_col,
            "std": std_col,
            "cutoff": np.inf,
            "train_removed": 0,
            "test_removed": 0
        }
        continue

    abs_z_train = np.abs((X_train[col] - mean_col) / std_col)
    cutoff = np.nanpercentile(abs_z_train, 99.9)

    # count before replacing
    train_outliers = (abs_z_train > cutoff).sum()

    # replace train
    X_train.loc[abs_z_train > cutoff, col] = np.nan

    abs_z_test = np.abs((X_test[col] - mean_col) / std_col)
    test_outliers = (abs_z_test > cutoff).sum()

    # replace test
    X_test.loc[abs_z_test > cutoff, col] = np.nan

    zscore_info[col] = {
        "mean": mean_col,
        "std": std_col,
        "cutoff": cutoff,
        "train_removed": int(train_outliers),
        "test_removed": int(test_outliers),
        "train_pct": train_outliers / len(X_train) * 100,
        "test_pct": test_outliers / len(X_test) * 100,
    }

# Convert to readable table
zscore_summary = pd.DataFrame(zscore_info).T

print("\nOutlier removal summary:")
display(zscore_summary.sort_values("train_removed", ascending=False))




Outlier removal summary:


,mean,std,cutoff,train_removed,test_removed,train_pct,test_pct
balance,1365.493420,3068.501079,11.141140,37.0,3.0,0.102300,0.033175
duration,258.506940,259.138862,7.285615,37.0,7.0,0.102300,0.077408
age,40.892999,10.626928,3.962293,36.0,5.0,0.099536,0.055291
previous,0.581730,2.408732,8.891927,36.0,9.0,0.099536,0.099524
pdays,40.157238,100.161229,6.098595,36.0,8.0,0.099536,0.088466
campaign,2.763935,3.104118,9.418478,29.0,9.0,0.080181,0.099524
day,15.817961,8.331865,1.822166,0.0,0.0,0.000000,0.000000


In [23]:
missing_before_imputation_train = X_train.isna().sum().sum()
missing_before_imputation_test = X_test.isna().sum().sum()

print(f"\nMissing values in train before imputation: {missing_before_imputation_train}")
print(f"Missing values in test before imputation:  {missing_before_imputation_test}")


Missing values in train before imputation: 41902
Missing values in test before imputation:  10474


In [24]:
# Split numeric and categorical parts
X_train_num = X_train[numerical_cols].copy()
X_test_num = X_test[numerical_cols].copy()

X_train_cat = X_train[categorical_cols].copy()
X_test_cat = X_test[categorical_cols].copy()

# Impute numeric columns with KNN
num_imputer = KNNImputer(n_neighbors=5, weights="distance")

X_train_num_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_train_num),
    columns=numerical_cols,
    index=X_train_num.index,
)

X_test_num_imputed = pd.DataFrame(
    num_imputer.transform(X_test_num),
    columns=numerical_cols,
    index=X_test_num.index,
)

# Impute categorical columns with most frequent
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat_imputed = pd.DataFrame(
    cat_imputer.fit_transform(X_train_cat),
    columns=categorical_cols,
    index=X_train_cat.index,
)

X_test_cat_imputed = pd.DataFrame(
    cat_imputer.transform(X_test_cat),
    columns=categorical_cols,
    index=X_test_cat.index,
)


In [25]:
from sklearn.preprocessing import OneHotEncoder

# One-hot encode categorical columns
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

X_train_cat_encoded = encoder.fit_transform(X_train_cat_imputed)
X_test_cat_encoded = encoder.transform(X_test_cat_imputed)


enconded_cols = encoder.get_feature_names_out(categorical_cols)

X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded,
    columns=enconded_cols,
    index=X_train_cat_imputed.index,
)

X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded,
    columns=enconded_cols,
    index=X_test_cat_imputed.index,
)

#X_train_cat_encoded.head(3)

In [26]:
# Recombine processed features
X_train_processed = pd.concat([X_train_num_imputed, X_train_cat_encoded], axis=1)
X_test_processed = pd.concat([X_test_num_imputed, X_test_cat_encoded], axis=1)

# Restore original column order
#X_train_processed = X_train_processed[X.columns]
#X_test_processed = X_test_processed[X.columns]

#print(f"Missing values in train after imputation: {X_train_processed.isna().sum().sum()}")
#print(f"Missing values in test after imputation:  {X_test_processed.isna().sum().sum()}")

In [27]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# copy to avoid overwriting
X_train_scaled = X_train_processed.copy()
X_test_scaled = X_test_processed.copy()

# numerical columns only
num_cols = X_train_scaled.select_dtypes(include=["number"]).columns

# Standard scaling
std_scaler = StandardScaler()

X_train_scaled[num_cols] = std_scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols] = std_scaler.transform(X_test_scaled[num_cols])

# MinMax scaling (after standard OR instead of — depends what you want)
minmax_scaler = MinMaxScaler()

X_train_scaled[num_cols] = minmax_scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols] = minmax_scaler.transform(X_test_scaled[num_cols])

X_train_scaled.head(3)

,age,balance,day,duration,campaign,pdays,previous,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success
24001,0.276923,0.204670,0.933333,0.065759,0.032258,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43409,0.092308,0.279923,0.133333,0.426022,0.096774,0.285276,0.318182,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20669,0.400000,0.190449,0.366667,0.814937,0.096774,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
# save data for next steps
X_train_scaled.to_csv("data/X_train_processed.csv", index=False)
X_test_scaled.to_csv("data/X_test_processed.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)